In [ ]:
# Map to 3D coordinates
# This script maps 2D coordinates to 3D coordinates based on a predefined mapping.

import pyvista as pv
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.interpolate import griddata
import os
from scipy.spatial import Delaunay
import csv


def read_xml_point_file(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Find all <point> entries under the first <time_series>
    points = []
    for point in root.findall('.//point_set/time_series/point'):
        x = float(point.find('x').text)
        y = float(point.find('y').text)
        z = float(point.find('z').text)
        points.append([x, y, z])

    return np.array(points)
def split_mesh_by_elemTag(mesh, tag_name='elemTag', onpts=True, tag_values=[0,3]):
    """
    Splits a PyVista mesh into RV free wall and septum based on an element tag.

    Parameters:
    - mesh: pyvista.UnstructuredGrid or PolyData
    - tag_name: name of the cell array to use for splitting (default 'elemTag')

    Returns:
    - rv_freewall_mesh: mesh with elemTag == 0
    - septum_mesh: mesh with elemTag == 3
    """    
    if tag_name not in mesh.cell_data:
        print(f"Tag '{tag_name}' not found in mesh cell data. Returning all indices")
        rv_freewall_indices = np.arange(mesh.n_points) 
        septum_indices = np.arange(mesh.n_points) 
        return rv_freewall_indices, septum_indices
     
    if onpts: # If onpts is True, we will average the elemTag across points
        elem_tags = mesh.cell_data["elemTag"]
        point_tags = [[] for _ in range(mesh.n_points)]

        for cell_id in range(mesh.n_cells):
            for pt_id in mesh.get_cell(cell_id).point_ids:
                point_tags[pt_id].append(elem_tags[cell_id])

        # Average or first
        avg_point_tags = np.array([np.mean(t) if t else -1 for t in point_tags], dtype=int)
        #print(f'Average point tags number of points: {avg_point_tags.shape[0]}')
        mesh.point_data["elemTag"] = avg_point_tags
        
        rv_freewall_indices = np.where(avg_point_tags == tag_values[0])[0] # elemTag = 0
        septum_indices = np.where(avg_point_tags == tag_values[1])[0] # elemTag = 3
        
        #print(f'RV Free Wall number of points: {rv_freewall_indices.shape[0]}')
        #print(f'Septum number of points: {septum_indices.shape[0]}')

        # Remove overlapping indices from septum_indices
        septum_indices = np.setdiff1d(septum_indices, rv_freewall_indices, assume_unique=False)
        #print(f'Septum number of points after removing RV Free Wall: {septum_indices.shape[0]}')

    else: 
        #rv_freewall_mesh = mesh.threshold(value=0.1, scalars=tag_name, invert=True)  # elemTag == 0
        #septum_mesh = mesh.threshold(value=(2.9, 3.1), scalars=tag_name)            # elemTag == 3
        
        rv_freewall_indices = np.where(mesh.cell_data[tag_name] == tag_values[0])[0] # elemTag == 0
        septum_indices = np.where(mesh.cell_data[tag_name] == tag_values[1])[0]   # elemTag == 3
        # rv_freewall_mesh = mesh.extract_cells(rv_freewall_indices)
        # septum_mesh = mesh.extract_cells(septum_indices)

    if septum_indices.size == 0:
        print(f"Error: No points found with elemTag = {tag_values}")
        rv_freewall_indices = np.arange(mesh.n_points) 
        septum_indices = np.arange(mesh.n_points) 
        return rv_freewall_indices, septum_indices
     

    return rv_freewall_indices, septum_indices

def normalize(v):
    return v / np.linalg.norm(v)

def scale_angles(rv0_theta, rv1_theta, theta_rel):
    """
    Map angles from full 0-360 range with specific reference points:
    - rv0_theta maps to 210°
    - rv1_theta maps to 330° 
    - Remaining angles are compressed into the other parts of the circle
    
    Parameters:
    rv0_theta: First reference angle (maps to 210°)
    rv1_theta: Second reference angle (maps to 330°)
    theta_rel: Input angle(s) to be mapped (0-360°)
    
    Returns:
    theta_final: Mapped angle(s) in the range [0, 360)
    """
    
    # Calculate the angular span between reference points
    delta_theta = (rv1_theta - rv0_theta) % 360
    
    # Handle theta_rel as array or scalar
    theta_rel = np.atleast_1d(theta_rel)
    theta_final = np.zeros_like(theta_rel, dtype=float)
    
    # Target span: rv0→210°, rv1→330° gives us 120° for the reference range
    target_span = 120  # from 210° to 330°
    
    # Remaining angular space to compress: 360° - 120° = 240°
    remaining_space = 240
    remaining_input_space = 360 - delta_theta
    
    for i, theta in enumerate(theta_rel):
        # Normalize theta relative to rv0
        theta_norm = (theta - rv0_theta) % 360
        
        if theta_norm <= delta_theta:
            # Angle is within the reference range [rv0, rv1]
            # Map linearly to [210°, 330°]
            progress = theta_norm / delta_theta if delta_theta > 0 else 0
            theta_final[i] = 210 + progress * target_span
        else:
            # Angle is outside the reference range
            # Compress into the remaining space
            remaining_theta = theta_norm - delta_theta
            compression_ratio = remaining_space / remaining_input_space if remaining_input_space > 0 else 1
            compressed_angle = remaining_theta * compression_ratio
            
            # Map to the space from 330° to 210° (wrapping around)
            theta_final[i] = (330 + compressed_angle) % 360
    
    # Return scalar if input was scalar
    if len(theta_final) == 1:
        return theta_final[0]
    return theta_final

def convert_to_polar_2d(mesh_points, pts, scale_z=True, verbose=False, RV_sept=True):
    apex = np.array(pts[0])
    base = np.array(pts[1])
    rv0 = np.array(pts[2])
    rv1 = np.array(pts[3])

    z_axis = normalize(base - apex)
    relative_points = mesh_points - apex
    base_dist = np.linalg.norm(base - apex)

    z_components = np.dot(relative_points, z_axis)[:, np.newaxis] * z_axis
    radial_vectors = relative_points - z_components

    z_coords = np.dot(relative_points, z_axis)
    apex_z_location= np.dot(apex-apex, z_axis)
    base_z_location = np.dot(base-apex, z_axis)                         
                             
    if verbose: 
        print('Z coordinates range:', np.min(z_coords), np.max(z_coords))
        print('Apex z location:', apex_z_location, 'Base z location:', base_z_location)
    if scale_z:
        # Normalize z_coords to [0, 1]
        z_coords = (z_coords - np.min(z_coords)) / (np.max(z_coords) - np.min(z_coords))
    else: 
        # scale to apex base distance
        z_coords = z_coords / base_dist

    ref_vector = normalize(rv0 - apex - np.dot((rv0 - apex), z_axis) * z_axis)
    y_ref = normalize(np.cross(z_axis, ref_vector))
    x_ref = ref_vector

    x_comp = np.dot(radial_vectors, x_ref)
    y_comp = np.dot(radial_vectors, y_ref)
    theta = np.degrees(np.arctan2(y_comp, x_comp)) % 360
    if verbose:
        print('Theta range:', np.min(theta), np.max(theta))
    # Get rv0 and rv1 current angles
    rv0_vec = rv0 - apex - np.dot((rv0 - apex), z_axis) * z_axis
    rv1_vec = rv1 - apex - np.dot((rv1 - apex), z_axis) * z_axis

    rv0_x = np.dot(rv0_vec, x_ref)
    rv0_y = np.dot(rv0_vec, y_ref)
    rv1_x = np.dot(rv1_vec, x_ref)
    rv1_y = np.dot(rv1_vec, y_ref)

    rv0_theta = np.degrees(np.arctan2(rv0_y, rv0_x)) % 360
    rv1_theta = np.degrees(np.arctan2(rv1_y, rv1_x)) % 360

    # Compute angular separation
    delta_theta = (rv1_theta - rv0_theta) % 360
    scale_factor = 120 / delta_theta  # from angle 210-330 scale to be 120 degrees

    # Remap full circle
    theta_rel = (theta - rv0_theta) % 360
    # If theta_rel is >300, then subtract 360
    theta_rel[theta_rel> 360] -= 360

    # If for the RV septal face 
    if RV_sept: 
        theta_scaled = (theta_rel * scale_factor) % 360
        theta_final = (theta_scaled + 210) % 360 # rotate to start at 210 
    else: # if for the full LV mapping 
    #     if theta_rel< delta_theta: 
    #         theta_scaled = (theta_rel * scale_factor) % 360
    #     else    
    #         scale_factor2 = 240/(360-delta_theta)
    #         theta_scaled = (theta_rel * scale_factor2)

    #     theta_final=theta_scaled + 120
    #     theta_final[theta_final> 360] -= 360

        theta_final=scale_angles(rv0_theta, rv1_theta, theta_rel)
        
    if verbose:
        print('rv0 theta:', rv0_theta, 'rv1 theta:', rv1_theta)
        print('Min theta:', np.min(theta_rel), 'Max theta:', np.max(theta_rel))
        print(f"rv0 θ = {rv0_theta:.2f}° → 210°, rv1 θ = {rv1_theta:.2f}° → 330°")
        print(f"Angular separation = {delta_theta:.2f}°, scale = {scale_factor:.4f}")
        print(f"Theta range: {np.min(theta_final):.2f}° to {np.max(theta_final):.2f}°")

    polar_coords = np.vstack((theta_final, z_coords)).T
    return polar_coords
def create_segment_mask(angle0, angle1, radius0, radius1, theta_grid, radius_grid):
    """
    Create a boolean mask for a segment defined by angular and radial bounds
    
    Parameters:
    angle0, angle1: Angular bounds in degrees
    radius0, radius1: Radial bounds (inner, outer)
    theta_grid, radius_grid: Coordinate grids (in degrees and normalized radius)
    
    Returns:
    mask: Boolean array where True indicates points within the segment
    """
    
    # Handle wraparound case (e.g., 330-360 and 0-30 for anterior segments)
    if angle0 > angle1:  # This shouldn't happen with our corrected data structure
        angle_mask = (theta_grid >= angle0) | (theta_grid <= angle1)
    else:
        angle_mask = (theta_grid >= angle0) & (theta_grid <= angle1)
    
    # Radial mask
    radius_mask = (radius_grid >= radius0) & (radius_grid <= radius1)
    
    # Combine masks
    mask = angle_mask & radius_mask
    
    return mask


def map_AHA_regions(csv_file, septum_mesh, field_str=None, CSP=False):
    """
    Map AHA (American Heart Association) 17-segment model values to mesh points.
    
    Parameters:
    csv_file: Path to CSV file with region values, or None for default mapping
    septum_mesh: Mesh object with point_data containing 'z_coords' and 'theta'
    field_str: String identifier for the field being mapped
    
    Returns:
    septum_mesh: Updated mesh with new field containing mapped values
    """
    
    # Radial boundaries (normalized: 0 = center, 1 = edge)
    apical_radius = 0.33   # Inner third (apex)
    mid_radius = 0.67      # Middle third  
    basal_radius = 1.0     # Outer third (base)
    
    # AHA 17-segment model definition
    # Note: Segments 1, 7, and 13 cross the 0°/360° boundary
    # segment_labels = {
    #     # Basal segments (1-6)
    #     1: [(0, 30, mid_radius, basal_radius), (330, 360, mid_radius, basal_radius)],    # Basal anterior
    #     2: [(270, 330, mid_radius, basal_radius)],   # Basal anteroseptal  
    #     3: [(210, 270, mid_radius, basal_radius)],   # Basal inferoseptal
    #     4: [(150, 210, mid_radius, basal_radius)],   # Basal inferior
    #     5: [(90, 150, mid_radius, basal_radius)],    # Basal inferolateral
    #     6: [(30, 90, mid_radius, basal_radius)],     # Basal anterolateral
        
    #     # Mid-cavity segments (7-12)
    #     7: [(0, 30, apical_radius, mid_radius), (330, 360, apical_radius, mid_radius)],  # Mid anterior
    #     8: [(270, 330, apical_radius, mid_radius)],   # Mid anteroseptal
    #     9: [(210, 270, apical_radius, mid_radius)],   # Mid inferoseptal  
    #     10: [(150, 210, apical_radius, mid_radius)],  # Mid inferior
    #     11: [(90, 150, apical_radius, mid_radius)],   # Mid inferolateral
    #     12: [(30, 90, apical_radius, mid_radius)],    # Mid anterolateral
        
    #     # Apical segments (13-16)
    #     13: [(315, 360, 0, apical_radius), (0, 45, 0, apical_radius)],  # Apical anterior
    #     14: [(225, 315, 0, apical_radius)],   # Apical septal
    #     15: [(135, 225, 0, apical_radius)],   # Apical inferior  
    #     16: [(45, 135, 0, apical_radius)],    # Apical lateral
    # }
    segments = {
        # Basal segments (1–6)
        1: {'theta_range': [[0, 30], [330, 360]], 'z_range': [mid_radius, basal_radius], 'label': 'Basal anterior', 'color': '#FFCCCC', 'txtcol': '#990000'},
        2: {'theta_range': [[270, 330]], 'z_range': [mid_radius, basal_radius], 'label': 'Basal anteroseptal', 'color': '#FFCC99', 'txtcol': '#994C00'},
        3: {'theta_range': [[210, 270]], 'z_range': [mid_radius, basal_radius], 'label': 'Basal inferoseptal', 'color': '#FFFF99', 'txtcol': '#999900'},
        4: {'theta_range': [[150, 210]], 'z_range': [mid_radius, basal_radius], 'label': 'Basal inferior', 'color': '#CCFFCC', 'txtcol': '#006600'},
        5: {'theta_range': [[90, 150]], 'z_range': [mid_radius, basal_radius], 'label': 'Basal inferolateral', 'color': '#99CCFF', 'txtcol': '#003366'},
        6: {'theta_range': [[30, 90]], 'z_range': [mid_radius, basal_radius], 'label': 'Basal anterolateral', 'color': '#CC99FF', 'txtcol': '#330066'},

        # Mid-cavity segments (7–12)
        7: {'theta_range': [[0, 30], [330, 360]], 'z_range': [apical_radius, mid_radius], 'label': 'Mid anterior', 'color': '#FF9999', 'txtcol': '#660000'},
        8: {'theta_range': [[270, 330]], 'z_range': [apical_radius, mid_radius], 'label': 'Mid anteroseptal', 'color': '#FFB266', 'txtcol': '#663300'},
        9: {'theta_range': [[210, 270]], 'z_range': [apical_radius, mid_radius], 'label': 'Mid inferoseptal', 'color': '#FFFF66', 'txtcol': '#666600'},
        10: {'theta_range': [[150, 210]], 'z_range': [apical_radius, mid_radius], 'label': 'Mid inferior', 'color': '#99FF99', 'txtcol': '#006633'},
        11: {'theta_range': [[90, 150]], 'z_range': [apical_radius, mid_radius], 'label': 'Mid inferolateral', 'color': '#6699FF', 'txtcol': '#003366'},
        12: {'theta_range': [[30, 90]], 'z_range': [apical_radius, mid_radius], 'label': 'Mid anterolateral', 'color': '#9966FF', 'txtcol': '#330066'},

        # Apical segments (13–16)
        13: {'theta_range': [[315, 360], [0, 45]], 'z_range': [0, apical_radius], 'label': 'Apical anterior', 'color': '#FF6666', 'txtcol': '#660000'},
        14: {'theta_range': [[225, 315]], 'z_range': [0, apical_radius], 'label': 'Apical septal', 'color': '#FFCC66', 'txtcol': '#664400'},
        15: {'theta_range': [[135, 225]], 'z_range': [0, apical_radius], 'label': 'Apical inferior', 'color': '#FFFF33', 'txtcol': '#666600'},
        16: {'theta_range': [[45, 135]], 'z_range': [0, apical_radius], 'label': 'Apical lateral', 'color': '#66FF99', 'txtcol': '#006633'},
        20: {'theta_range': [[260, 280]], 'z_range': [0.45, 0.89], 'label': 'LBB', 'color': "#FF86FF", 'txtcol': "#FF4DFF"}, # LB 
        21: {'theta_range': [[270, 330]], 'z_range': [0.5, 0.75], 'label': 'LAF', 'color': "#fdb56c", 'txtcol': '#cc6600'}, # LV anterior branch area
        22: {'theta_range': [[225, 270]], 'z_range': [0.38, 0.625], 'label': 'LPF', 'color': '#4da6ff', 'txtcol': "#003d7a"}, # LV posterior branch area 
        23: {'theta_range': [[252, 274]], 'z_range': [0.25, 0.6], 'label': 'LSF', 'color': '#66b38c', 'txtcol': "#00664f"}, # LV posterior branch area 
    }

    def create_segment_mask(angle0, angle1, radius0, radius1, theta, z_coords):
        """Create boolean mask for points within specified angular and radial bounds"""
        
        # Handle angle wrapping around 0°/360° boundary
        if angle0 <= angle1:
            angle_mask = (theta >= angle0) & (theta <= angle1)
        else:  # wraps around 0°/360°
            angle_mask = (theta >= angle0) | (theta <= angle1)
        
        # Create radial mask
        radius_mask = (z_coords >= radius0) & (z_coords <= radius1)
        
        # Combine masks
        return angle_mask & radius_mask

    # Initialize output array with NaN
    if CSP:
        assigned_values = np.full(septum_mesh.n_points, 0)
    else: 
        assigned_values = np.full(septum_mesh.n_points, np.nan)
    # Get mesh coordinates
    z_coords = septum_mesh.point_data['z_coords']
    theta = septum_mesh.point_data['theta']
    
    # Load data from CSV file or use defaults
    if csv_file is not None: 
        try:
            with open(csv_file, 'r') as f:
                reader = csv.reader(f)
                next(reader)  # Skip header
                region_stats = [row for row in reader]
                AHA_regions = [int(row[0]) for row in region_stats]
                values = [float(row[1]) for row in region_stats]  # mean values for each region
                field = f'mean_{field_str}'
                
        except FileNotFoundError: 
            print(f'File not found: {csv_file}')
            return septum_mesh
            
    else: 
        # Default mapping: assign region number as value
        
        if CSP: 
            field='CSP_mapping'
            AHA_regions=list(range(20,24))
            values=[20, 21, 22, 24]
        else: 
            field = 'AHA_mapping'
            AHA_regions = list(range(1, 17))  # [1, 2, 3, ..., 16]
            values = list(range(1, 17))       # [1, 2, 3, ..., 16]

    # Process each AHA region
    for i, region in enumerate(AHA_regions):
        if region not in segments:
            print(f'Warning: Region {region} not found in segment definitions')
            continue
            
        seg = segments[region]
        mean_value = values[i]
        
        # Loop over one or more theta ranges for this segment
        for theta_range in seg['theta_range']:
            #print(theta_range)
            angle0, angle1 = theta_range
            radius0, radius1 = seg['z_range']

            # Create mask for this angular + radial range
            region_mask = create_segment_mask(angle0, angle1, radius0, radius1, theta, z_coords)

            # Assign value to all points in this region
            if np.any(region_mask):
                if CSP: 
                    assigned_values[region_mask] += mean_value
                else: 
                    assigned_values[region_mask] = mean_value

        # # Process each angular range for this segment (handles wraparound cases)
        # for angle0, angle1, radius0, radius1 in segment_ranges:
        #     region_mask = create_segment_mask(angle0, angle1, radius0, radius1, theta, z_coords)
            
        #     # Assign value to all points in this region
        #     if np.any(region_mask):
        #         assigned_values[region_mask] = mean_value

    # Add the new field to mesh
    
    septum_mesh.point_data[field] = assigned_values
    
    return septum_mesh
def AHAmap_3D(mainpath, xml_file, refmesh, output_file,flip_flag=True, verbose=False, CSP=False): 

    try:
        points_array = read_xml_point_file(xml_file)
        if verbose: 
            print(f'Points array: {points_array}')
        if flip_flag: 
        # Flip 
            points_array[:,0]=-points_array[:,0]
            points_array[:,1]=-points_array[:,1]

        point_cloud = pv.PolyData(points_array)
        point_cloud.save(f'{mainpath}/1_seg/ap_mv_2rv_Siemens.vtk')
    except FileNotFoundError:
        print(f"XML file not found for case {casenum:02d}. Skipping this case.")
        return    
    
    print('Splitting mesh into LV indices')
    _, septum_indices = split_mesh_by_elemTag(refmesh, tag_name='elemTag',onpts=True, tag_values=[0, 1])


    septum_mesh = refmesh.extract_points(septum_indices, adjacent_cells=False)
    if verbose: 
        print(refmesh)
        print(f'Septum indices: {septum_indices}')
        septum_meshfile=f'{mainpath}/test.vtk'
        print(f'Save septum mesh to {septum_meshfile}')
        septum_mesh.save(septum_meshfile)

    septum_points = convert_to_polar_2d(septum_mesh.points,  points_array,scale_z=False, verbose=True, RV_sept=False)
    if verbose: 
        print(f'converted septum points: {septum_points}')
    septum_mesh.point_data['theta'] = septum_points[:, 0]
    septum_mesh.point_data['z_coords'] = septum_points[:, 1]

    septum_mesh=map_AHA_regions(csv_file=None, septum_mesh=septum_mesh, CSP=CSP)

    
    print(f'Saving septum mesh for case {casenum:02d} to {output_file}')
    septum_mesh.save(output_file)
    return

if __name__ == "__main__":   
    for casenum in [1]: 
        if casenum in [3, 5, 12]: 
            continue
        mainpath=f'/data_fast/Work/MS_HF_cohort/data/{casenum:02d}'
        # LV mesh
        LV_refmesh=pv.read(f'{mainpath}/LV.vtk')
        LV_output_file= f'{mainpath}/1_seg/LV_AHA.vtk'
    
        # RV septum mesh 
        RV_refmesh=pv.read(f'{mainpath}/1_seg/RV_smooth_flip.vtk')
        RV_output_file= f'{mainpath}/1_seg/sept_AHA.vtk'
        
        

        xml_file = f'{mainpath}/1_seg/ap_mv_2rv_Siemens.mps'

        #AHAmap_3D(xml_file, LV_refmesh, LV_output_file, flip_flag=True)
        #AHAmap_3D(mainpath, xml_file, RV_refmesh, RV_output_file, flip_flag=True, verbose=True)
        

In [ ]:
def get_mesh_output(device, length, collision_field= 'Combined_Collision',CSP=False): 
    all_interp_fields_RV_freewall = []
    all_interp_fields_RV_septum = []
    for casenum in range(1,14): #[1:13]:
        if casenum in [3, 5, 12]: # Skip cases with no data
            print(f'Skipping case {casenum:02d} due to no data.')
            continue
        mainpath=f'/data_fast/Work/MS_HF_cohort/data/{casenum:02d}'
        output_file= f'{mainpath}/1_seg/RV_septum_polar.vtk'
        
        if collision_field == 'Combined_Collision':
            images_outpath=f'/data_fast/Work/MS_HF_cohort/images/{device}/'
            add_legend=True
        else: 
            images_outpath=f'/data_fast/Work/MS_HF_cohort/images/{device}/{collision_field}/'
            add_legend=False
        
        xml_file = f'{mainpath}/1_seg/ap_mv_2rv_Siemens.mps'
        try:
            points_array = read_xml_point_file(xml_file)
            print(points_array)
        except FileNotFoundError:
            print(f"XML file not found for case {casenum:02d}. Skipping this case.")
            continue    
                
        # Read the septum mesh if it exists, otherwise create it
        if os.path.exists(output_file):
            septum_mesh = pv.read(output_file)
            print(f'Loading existing septum mesh for case {casenum:02d} from {output_file}')
            
        else: 
            init_mesh=f'{mainpath}/1_seg/RV_smooth.vtk'
            print(f'Processing case {casenum:02d} with mesh {init_mesh}')
            refmesh=pv.read(init_mesh)
            rv_freewall_indices, septum_indices = split_mesh_by_elemTag(refmesh, tag_name='elemTag',onpts=True)
            septum_mesh = refmesh.extract_points(septum_indices, adjacent_cells=False)

        # Process the septum mesh
        septum_points = convert_to_polar_2d(septum_mesh.points, points_array)
        
        septum_mesh.point_data['theta'] = septum_points[:, 0]
        septum_mesh.point_data['z_coords'] = septum_points[:, 1]

        csv_file=f'{images_outpath}/sum_plots/RV_septum_polar_sum_{device}_L{length}.csv'
        #print(f'Loading existing CSV file for case {casenum:02d} from {csv_file}')
        field_str= f'{device}{length}_{collision_field.replace('_Collision', '')}'
        septum_mesh=map_AHA_regions(csv_file, septum_mesh, field_str=field_str, CSP=CSP)

        print(f'Saving septum mesh for case {casenum:02d} to {mainpath}/1_seg/RV_septum_polar.vtk')
        septum_mesh.save(output_file)
        #return septum_mesh
        # add in the points from the xml file 
        #lm_points= convert_to_polar_2d(points_array, points_array)
        return 
def collmap_3D():
    # Map to AHA
    # AveirVR 38mm, AveirAR 32.2mm, Micra 25.9mm
    ########################################################
    for device in ['AveirVR', 'AveirAR', 'Micra']: 
        if device == 'AveirVR':
            init_len= 38
        elif device == 'AveirAR': 
            init_len= 32.2
        elif device == 'Micra': 
            init_len = 25.9
        else: 
            print(f'WARNING - DEVICE: {device} is not AveirVR, AveirAR or Micra')        


        for length in [init_len]: #, 15, 20, 25, 30, 35, 40, 45]:  #[22, 27, 32, 37, 42]: 
            for collision_field in ['RV_Collision', 'TV_Collision', 'PM_Collision', 'Combined_Collision']: #['Combined_Collision']: #
                get_mesh_output(device, length, collision_field=collision_field)
if __name__ == "__main__":
    # To map the averaged collision map value on the AHA mapped onto the 3D mesh
    collmap_3D()

[[ 73.4769  71.7324 360.213 ]
 [ 25.1018 135.253  377.43  ]
 [ 17.22   110.552  338.937 ]
 [ 44.8454  96.6719 408.836 ]]
Loading existing septum mesh for case 01 from /data_fast/Work/MS_HF_cohort/data/01/1_seg/RV_septum_polar.vtk
Saving septum mesh for case 01 to /data_fast/Work/MS_HF_cohort/data/01/1_seg/RV_septum_polar.vtk
[[ 73.4769  71.7324 360.213 ]
 [ 25.1018 135.253  377.43  ]
 [ 17.22   110.552  338.937 ]
 [ 44.8454  96.6719 408.836 ]]
Loading existing septum mesh for case 01 from /data_fast/Work/MS_HF_cohort/data/01/1_seg/RV_septum_polar.vtk
Saving septum mesh for case 01 to /data_fast/Work/MS_HF_cohort/data/01/1_seg/RV_septum_polar.vtk
[[ 73.4769  71.7324 360.213 ]
 [ 25.1018 135.253  377.43  ]
 [ 17.22   110.552  338.937 ]
 [ 44.8454  96.6719 408.836 ]]
Loading existing septum mesh for case 01 from /data_fast/Work/MS_HF_cohort/data/01/1_seg/RV_septum_polar.vtk
Saving septum mesh for case 01 to /data_fast/Work/MS_HF_cohort/data/01/1_seg/RV_septum_polar.vtk
[[ 73.4769  71.7324

In [2]:
# Visualize the mesh
import numpy as np
import pyvista as pv    
import vtk
from vtkmodules.vtkRenderingAnnotation import vtkScalarBarActor
from vtkmodules.vtkCommonCore import vtkStringArray, vtkDoubleArray
import matplotlib as plt
def set_scalar_bar_mid_labels(plotter, n_labels=5):
    """
    Replace scalar bar tick labels with midpoints between bins.
    
    Parameters
    ----------
    plotter : pyvista.Plotter
        The plotter containing the scalar bar.
    n_labels : int
        Number of midpoint labels to show.
    """
    if not plotter.scalar_bars:
        raise RuntimeError("No scalar bar found in the plotter.")
    
    # Grab the first scalar bar actor
    scalar_bar: vtkScalarBarActor = list(plotter.scalar_bars.values())[0]
    
    # Get scalar range
    #vmin, vmax = scalar_bar.GetLookupTable().GetRange()
    #print(vmin, vmax)
    # Compute bin centers
    #edges = np.linspace(1, 16, n_labels + 1)
    #centers = 0.5 * (edges[:-1] + edges[1:])
    centers = np.arange(1.4, 17.4, 1) # arange goes to 1 below
    #print(centers)
    # Convert to vtk array
    ticks = vtkDoubleArray()
    for c in centers:
        ticks.InsertNextValue(float(c))
    #print(ticks)
    #print(centers)
    # Apply custom tick positions
    scalar_bar.SetCustomLabels(ticks)
    scalar_bar.SetUseCustomLabels(True)
    scalar_bar.SetNumberOfLabels(len(centers))
    scalar_bar.SetLabelFormat('%.0f')
    #scalar_bar.label_text_property.SetFontSize(14)
    #scalar_bar.title_text_property.SetFontSize(16)

def mycolormap(): 

    colors = [
        "#4da6ff",  # your blue
        "#66b38c",  # your teal
        "#ff9933",  # your orange
        "#ffd92f",  # yellow
        "#a6d854",  # lime green
        "#8da0cb",  # soft blue
        "#e78ac3",  # pink
        "#fc8d62",  # salmon
        "#b3b3b3",  # gray
        "#1b9e77",  # strong green
        "#d95f02",  # strong orange
        "#7570b3",  # purple
        "#e7298a",  # magenta
        "#66a61e",  # dark green
        "#e6ab02",  # mustard
        "#a6761d",  # brown
    ]

    
    cmap = ListedColormap(colors[:16], name="colorblind_rainbow")
    return cmap
def set_camera_position(plotter):
    # 
    # view = GetActiveView()
    # print("CameraPosition:", view.CameraPosition)
    # print("CameraFocalPoint:", view.CameraFocalPoint)
    # print("CameraViewUp:", view.CameraViewUp)
    # print("CameraParallelScale:", view.CameraParallelScale)

    plotter.camera_position = [
        (152.42287108591592, 123.59827412877627, 656.7684240299902), 
        (-29.384029209520445, -93.8310364745493, 379.2673664416711),
        (0.6384333843111966, -0.7506886833483885, 0.16990972452927794)
    ]
    plotter.enable_parallel_projection()
    plotter.camera.parallel_scale = 102.66216262784093
def plot_meshes(plotter, biv_masked_mesh,septum_mesh,scalars='mean_AveirVR38_RV',screenshot_path=None, cmap='PuBu', clim=[0, 100], n_colors=20,  nan_color=[0,0,0,0], scalar_bar_title='RV Septum Mean Values'): 
    
#cmap='coolwarm',
    #shifted_data=scalars+0.5
    plotter.add_mesh(septum_mesh, scalars=scalars, cmap=cmap, show_edges=False,                 
                     name=scalar_bar_title, clim=clim, n_colors=n_colors, 
                     show_scalar_bar=True, nan_color=nan_color, interpolate_before_map=False, opacity=1)

    set_scalar_bar_mid_labels(plotter, n_labels=n_colors)   
    plotter.add_mesh(biv_masked_mesh, color='lightgray', opacity=0.1, show_edges=False, name='BiV_Masked')
    set_camera_position(plotter)

    return plotter


def get_collision_map_screenshots():     

    for case in [1]: #,2,4,6,7,8,9,10,11,13]: #range(1,14): #[1:13]:
        mainpath=f'/data_fast/Work/MS_HF_cohort/data/{case:02d}'  
        # Read the septum mesh
        septum_mesh = pv.read(f'{mainpath}/1_seg/RV_septum_polar.vtk')
        # Read in BiV_masked mesh 
        biv_masked_mesh = pv.read(f'{mainpath}/1_seg/BiV_masked.vtk') 

        for device in ['Micra']: # 'AveirVR', 'AveirAR', 
            if device == 'AveirVR':
                init_len= 38
            elif device == 'AveirAR': 
                init_len= 32.2
            elif device == 'Micra': 
                init_len = 25.9
            else: 
                print(f'WARNING - DEVICE: {device} is not AveirVR, AveirAR or Micra')        

                #, RV_Collision, TV_Collision, 'PM_Collision', 'Combined_Collision'
            
            for length in [init_len]: #, 15, 20, 25, 30, 35, 40, 45]:  #[22, 27, 32, 37, 42]: 
                for collision_field in ['Combined_Collision']: #['Combined_Collision']: #
                    field_str= f'{device}{length}_{collision_field.replace("_Collision", "")}'
                    print(f'Plotting for device {device}, length {length}, collision field {collision_field}')
                    screenshot_path = f'/data_fast/Work/MS_HF_cohort/images/3D_output/case{case:02d}_RV_septum_polar_sum_{field_str}.png'
                    plot_meshes(biv_masked_mesh, septum_mesh, scalars=f'mean_{field_str}', screenshot_path=screenshot_path)

        #plot_meshes(biv_masked_mesh,septum_mesh,scalars='mean_AveirVR38_RV',screenshot_path=None)

def get_AHA_map_screenshots(mainpath, septum_file='1_seg/LV_AHA.vtk', scalars='AHA_mapping', screenshot_path=None, addpm=False, addtv=False, addrv=False, addcy=False, addlv=False, addCS=False, addTVring=False, addCSP_fields=None, addBIV=False):     
        biv_masked_mesh = pv.read(f'{mainpath}/2_motion/BiV_masked_flip.vtk') 
        # Read in BiV_masked mesh 
        if addBIV: 
            
            rv_sept_mesh = pv.read(f'{mainpath}/1_seg/RV_septum_polar.vtk') 
        if addtv:   
            tv_mesh=pv.read(f'{mainpath}/2_motion/TV_flip.vtk')
        if addpm: 
            pm_mesh=pv.read(f'{mainpath}/2_motion/PM_flip.vtk')
        if addrv: 
            rv_mesh=pv.read(f'{mainpath}/2_motion/RV_smooth_flip.vtk')
        if addcy:
            cy_mesh=pv.read(f'{mainpath}/2_motion/cylinder200.vtp')
        if addlv: 
            if septum_file is None: 
                biv_masked_mesh = pv.read(f'{mainpath}/2_motion/BiV_masked_flip.vtk') 
            else: 
                lv_mesh=pv.read(f'{mainpath}/1_seg/LV_AHA.vtk')
        if addCSP_fields is not None: 
            # Dictionary to store glowing regions per field
            glowing_regions = {}
            for addCSP_field in addCSP_fields: 
                glowing_region=None
                CSP_mesh=pv.read(f'{mainpath}/1_seg/sept_AHA_CSP.vtk')
                valid_fields = ["LBB", "LAF", "LPF", "LSF"]  # priority order            
                if addCSP_field in CSP_mesh.point_data:        
                    glowing_region = CSP_mesh.threshold(value=20.0, scalars=addCSP_field) # LAF
                    glowing_regions[addCSP_field] = glowing_region
            
        if addCS: 
            #CS_upper=pv.read(f'{mainpath}/1_seg/CS_upper.vtk')
            LV_fas=pv.read(f'{mainpath}/1_seg/LB_ant_post.vtk')
            LB=pv.read(f'{mainpath}/1_seg/LBB.vtk')
        if addTVring: 
            TVring=pv.read(f'{mainpath}/1_seg/TV20_surface.vtk')
            #TVring=pv.read(f'{mainpath}/1_seg/TV_annulus-20_flip.vtk')
        ####################################################    
        plotter = pv.Plotter()
        set_camera_position(plotter) 


        if addBIV and not addlv:
            epi = biv_masked_mesh.threshold(value=(-0.5, 0.5), scalars="elemTag", invert=False)     
            lv_endo=biv_masked_mesh.threshold(value=(9.5, 10.5), scalars="elemTag", invert=False)     
            plotter.add_mesh(epi,color="white",opacity=0.1,specular=0.9,specular_power=20)
            rv_sept_mesh.translate([-1,-1,0], inplace=True)  # shifts all points by +1 in x, y, z
            plotter.add_mesh(rv_sept_mesh,color="#FFFFFF",opacity=1) #,specular=0.9,specular_power=20)
            if addtv or addrv or addpm: 
                plotter.add_mesh(lv_endo,color="#000069",opacity=0.3) # ,specular=0.9,specular_power=20)
            else:   
                plotter.add_mesh(lv_endo,color="#000069",opacity=0.3) # ,specular=0.9,specular_power=20)
            
        ### PLOT THE LV AHA mesh with the biv mesh 
        if addlv: 
            if septum_file is not None: 
                print(f'add the {mainpath}/1_seg/LV_AHA.vtk file First')
                cmap=mycolormap()
                #cmap='coolwarm' # cmap='PuBu'
                plotter.add_mesh(biv_masked_mesh, color='lightgray', opacity=0.1, show_edges=False, name='BiV_Masked')

                lv_mesh[scalars]=lv_mesh[scalars]+0.4        
                plotter.add_mesh(lv_mesh, scalars=scalars, cmap=cmap, show_edges=False,                 
                    clim=[1,17], n_colors=16, show_scalar_bar=False, nan_color='lightgray', interpolate_before_map=False, opacity=0.3)
                septum_mesh = pv.read(f'{mainpath}/{septum_file}')         
                plotter.add_mesh(septum_mesh, scalars=scalars, cmap=cmap, show_edges=False,                 
                    clim=[1,17], n_colors=16, show_scalar_bar=False, nan_color=[0,0,0,0], interpolate_before_map=False, opacity=1)
                
            elif addBIV: # Add the biv mesh when we don't have a septum_file 
                #plotter.add_mesh(biv_masked_mesh, opacity=0.1, color="white")
                plotter.add_mesh(
                    biv_masked_mesh,
                    color="white",                    
                    opacity=0.1,             # Transparency
                    specular=0.9,            # Strong reflections
                    specular_power=20,       # Shininess
                )
        # Read the septum mesh
        
        else: 
            if septum_file is not None:
                septum_mesh = pv.read(f'{mainpath}/{septum_file}')            
                septum_mesh[scalars]=septum_mesh[scalars]+0.4    
                epi=biv_masked_mesh.threshold(value=(-0.5, 0.5), scalars="elemTag", invert=False)
                plotter=plot_meshes(plotter, epi, septum_mesh, scalars=scalars, screenshot_path=None, cmap='coolwarm', clim=[1, 17], n_colors=16, scalar_bar_title='AHA regions', nan_color=[0.8,0.8,0.8,0.1])    
        if addpm: 
            plotter.add_mesh(pm_mesh, opacity=0.5, color="#66b38c" ) #"#ff9933")
        if addtv: 
            plotter.add_mesh(tv_mesh, opacity=0.5, color="#4da6ff") 
        if addrv: 
            plotter.add_mesh(rv_mesh, opacity=0.3, color="#fdb56c")
        if addcy: 
            plotter.add_mesh(cy_mesh, opacity=0.8, color="white") 
        if addCS: 
            # plotter.add_mesh(CS_upper,
			# 		 scalars="elemTag",
			# 		 show_edges=False,
			# 		 cmap=plt.cm.get_cmap("coolwarm",8),
			# 		 line_width=7,
			# 		 style='wireframe',
			# 		 render_lines_as_tubes=True)
            plotter.add_mesh(LB, show_edges=False, style='wireframe',line_width=10,render_lines_as_tubes=True, color="#F87CF8")
            plotter.add_mesh(LV_fas, show_edges=False, style='wireframe',line_width=5,render_lines_as_tubes=True, color="#6D00A0")
            # Get mesh center
            center = np.array(LV_fas.center)  # convert tuple → NumPy array
            center[1] += 10  # shift 10 units in Y direction
            if addCSP_fields is None: 
                # Add 3D text at the center of LV_fas
                plotter.add_point_labels(
                    [center],           # list of points
                    ["Left bundle branches"],         # corresponding labels                
                    point_size=10,
                    text_color="#6D00A0",
                    font_size=20,
                    shape_opacity=0.5
                )
        if addTVring: 
            #plotter.add_mesh(TVring, color="#FFA64D", point_size=10, render_points_as_spheres=True)
            plotter.add_mesh(TVring, show_edges=False, style='wireframe',line_width=10,render_lines_as_tubes=True, color="#fff32c")
            center = np.array(TVring.center)  # convert tuple → NumPy array
            center[0] += 10  # shift 10 units in Y direction
            center[1] += 2  # shift 10 units in Y direction

            # Add 3D text at the center of LV_fas
            plotter.add_point_labels(
                [center],           # list of points
                ["  20mm inferior \n to the TV annulus "],         # corresponding labels                
                point_size=10,
                text_color="#fff32c",
                font_size=20,
                shape_opacity=0.5
            )
        if addCSP_fields is not None and glowing_regions: 
            for addCSP_field in addCSP_fields: 
                CSP_colors={
                    'LBB' : {'color': "#FF86FF", 'txtcol': "#C714C7", 'xshift': 15, 'yshift': 0}, # LB 
                    'LAF' : {'color': "#fdb56c", 'txtcol': '#cc6600', 'opacity': 0.6}, # LV anterior branch area
                    'LPF' : {'color': '#4da6ff', 'txtcol': "#003d7a"}, # LV posterior branch area 
                    'LSF' : {'color': '#66b38c', 'txtcol': "#00664f", 'xshift': -10, 'yshift': 10} # LV posterior branch area 
                }
                txtcol=CSP_colors[addCSP_field]['txtcol']
                color=CSP_colors[addCSP_field]['color']

                glowing_region=glowing_regions[addCSP_field]
                center = np.array(glowing_region.center)  # convert tuple → NumPy array
                xshift = CSP_colors.get(addCSP_field, {}).get('xshift', 10)  # default 0
                yshift = CSP_colors.get(addCSP_field, {}).get('yshift', 10)  # default 0
                center[0] += xshift # shift 10 units in X direction
                center[1] += yshift # shift 10 units in Y direction

                # Add 3D text at the center of LV_fas
                plotter.add_point_labels(
                    [center],           # list of points
                    [addCSP_field],         # corresponding labels                
                    point_size=1,
                    text_color=txtcol,
                    font_size=20,
                    shape_opacity=0.5
                )
                plotter.add_mesh(
                    glowing_region,
                    color=color,     # neon-like colormap
                    opacity=CSP_colors.get(addCSP_field, {}).get('opacity', 0.5) ,
                    ambient=1.0,         # fully emissive
                    diffuse=0.0,
                    specular=0.9,        # slight shine for extra glow
                    specular_power=50,   # sharper highlight
                    render_points_as_spheres=True,
                    point_size=50,       # big for glow effect
                )
        if screenshot_path is not None: 
            plotter.screenshot(screenshot_path)
            print(f'Screenshot saved to {screenshot_path}')
            plotter.close()  # Close without showing
        else: 
            # Onlt show the plot if no screenshot path is provided
            plotter.show()
if __name__ == "__main__":
    #get_collision_map_screenshots()
    # Figure 1A 
    #for case in [1]: #,2,4,6,7,8,9,10,11,13]: #range(1,14): #[1:13]:
    mainpath=f'/data_fast/Work/MS_HF_cohort/data/01'  
    #screenshot_path = f'/data_fast/Work/MS_HF_cohort/images/3D_output/case01_BiV_masked_LV_AHA.png'
    screenshot_path=None
    #get_AHA_map_screenshots(mainpath, septum_file='1_seg/LV_AHA.vtk', scalars='AHA_mapping', screenshot_path = screenshot_path, addpm=True, addtv=True, addrv=False)

    # Fig1B 
    screenshot_path = f'/data_fast/Work/MS_HF_cohort/images/3D_output/case01_BiV_all_structures.png'
    #get_AHA_map_screenshots(mainpath, septum_file=None, scalars='AHA_mapping', screenshot_path = screenshot_path, addpm=False, addtv=False, addrv=False, addcy=False, addlv=True, addCS=True, addTVring=True)
    
    # Fig1C
    screenshot_path = f'/data_fast/Work/MS_HF_cohort/images/3D_output/TV_fig1C.png'
    #get_AHA_map_screenshots(mainpath, septum_file='1_seg/sept_AHA.vtk', scalars='AHA_mapping', screenshot_path = screenshot_path, addpm=True, addtv=True, addrv=False, addcy=True, addlv=True, addCS=False)

    # cy=cylinder
    screenshot_path = f'/data_fast/Work/MS_HF_cohort/images/3D_output/TV_fig1B.png'
    #get_AHA_map_screenshots(mainpath, septum_file=None, scalars='AHA_mapping', screenshot_path = screenshot_path, addpm=False, addtv=False, addrv=False, addcy=False, addlv=True, addCS=True, addTVring=True)
    
    
    # Figure - add LAF patch 
    screenshot_path=f'/data_fast/Work/MS_HF_cohort/images/3D_output/case01_CSP.png'
    #get_AHA_map_screenshots(mainpath, septum_file=None, scalars='AHA_mapping', screenshot_path = None, addpm=False, addtv=False, addrv=False, addcy=True, addlv=False, addCS=True, addTVring=False, addCSP_fields=['LBB', 'LPF', 'LSF', 'LAF'])
    ####################################################################################
    # Graphical Abstract
    # screenshot_path=f'/data_fast/Work/MS_HF_cohort/images/graph_abstract/GraphAb_B1.png'    #screenshot_path=None
    # get_AHA_map_screenshots(mainpath, septum_file=None, screenshot_path=screenshot_path, addBIV=True, addlv=False, scalars='elemTag', addCSP_fields=['LBB', 'LPF', 'LSF', 'LAF'])
    # screenshot_path=f'/data_fast/Work/MS_HF_cohort/images/graph_abstract/GraphAb_TV1.png'
    # get_AHA_map_screenshots(mainpath, septum_file=None, screenshot_path=screenshot_path, addBIV=True, addlv=False, scalars='elemTag', addtv=True)
    # screenshot_path=f'/data_fast/Work/MS_HF_cohort/images/graph_abstract/GraphAb_RV1.png'    
    # get_AHA_map_screenshots(mainpath, septum_file=None, screenshot_path=screenshot_path, addBIV=True, addlv=False, scalars='elemTag', addrv=True)
    # screenshot_path=f'/data_fast/Work/MS_HF_cohort/images/graph_abstract/GraphAb_PM1.png'    
    # get_AHA_map_screenshots(mainpath, septum_file=None,  screenshot_path=screenshot_path, addBIV=True, addlv=False, addpm=True)

    ####################################################################################
    # Cardiac Physiome 2026 abstract
    screenshot_path=f'/data_fast/Work/MS_HF_cohort/images/cardiacphysiome2026/3D_all.png'
    get_AHA_map_screenshots(mainpath, septum_file=None, screenshot_path=screenshot_path, addBIV=True, addlv=False, scalars='elemTag', addtv=True, addrv=True, addpm=True, addcy=True)

Screenshot saved to /data_fast/Work/MS_HF_cohort/images/cardiacphysiome2026/3D_all.png


In [17]:
import pyvista as pv
import numpy as np


def extract_septum_from_rv_mesh(input_file, output_file, septum_tag=3):
    """
    Extract septum region from RV mesh based on elemTag value.
    Works whether elemTag is stored in point_data or cell_data.
    
    Parameters:
    input_file: Path to input VTK file (e.g., 'RV_smooth_flip.vtk')
    output_file: Path to output VTK file (e.g., 'RV_septum_polar.vtk')
    septum_tag: Value in elemTag field that identifies septum (default: 3)
    """
    
    # Load the mesh
    print(f"Loading mesh from: {input_file}")
    rv_mesh = pv.read(input_file)
    
    # Display mesh info
    print(f"Original mesh:")
    print(f"  Points: {rv_mesh.n_points}")
    print(f"  Cells: {rv_mesh.n_cells}")
    print(f"  Point data arrays: {list(rv_mesh.point_data.keys())}")
    print(f"  Cell data arrays:  {list(rv_mesh.cell_data.keys())}")
    
    # Check where elemTag exists
    elem_source = None
    if 'elemTag' in rv_mesh.point_data:
        elem_source = 'point'
        elem_tags = rv_mesh.point_data['elemTag']
    elif 'elemTag' in rv_mesh.cell_data:
        elem_source = 'cell'
        elem_tags = rv_mesh.cell_data['elemTag']
    else:
        print("Error: 'elemTag' field not found in point_data or cell_data")
        return None
    
    print(f"\nUsing elemTag from: {elem_source}_data")
    unique_tags = np.unique(elem_tags)
    print(f"Unique elemTag values: {unique_tags}")
    
    # Count elements by tag
    for tag in unique_tags:
        count = np.sum(elem_tags == tag)
        print(f"  Tag {tag}: {count} {'points' if elem_source=='point' else 'cells'}")
    
    # Extraction
    print("\nExtracting septum mesh...")
    if elem_source == 'point':
        # Threshold on point_data
        #septum_mesh = rv_mesh.threshold(septum_tag - 0.5, septum_tag + 0.5, scalars='elemTag')
        septum_mesh = rv_mesh.threshold(value=(septum_tag - 0.5, septum_tag + 0.5), scalars='elemTag')
    else:
        # Threshold on cell_data
        #septum_mesh = rv_mesh.threshold(septum_tag - 0.5, septum_tag + 0.5, scalars='elemTag', preference='cell')
         septum_mesh = rv_mesh.threshold(value=(septum_tag - 0.5, septum_tag + 0.5), scalars='elemTag', preference='cell')
    if septum_mesh.n_points == 0:
        print("Error: Extraction failed (no points).")
        return None
    
    # Verify
    print(f"\nFinal septum mesh:")
    print(f"  Points: {septum_mesh.n_points}")
    print(f"  Cells: {septum_mesh.n_cells}")
    if 'elemTag' in septum_mesh.point_data:
        print(f"  Extracted point elemTags: {np.unique(septum_mesh.point_data['elemTag'])}")
    if 'elemTag' in septum_mesh.cell_data:
        print(f"  Extracted cell elemTags: {np.unique(septum_mesh.cell_data['elemTag'])}")
    
    # Save
    print(f"\nSaving septum mesh to: {output_file}")

    septum_mesh = septum_mesh.extract_surface()  # Converts to PolyData
    septum_mesh.save(output_file, binary=False)

    print("✓ Septum mesh saved successfully!")
    
    return septum_mesh

# Main execution
if __name__ == "__main__":
    mainpath='/data_fast/Work/MS_HF_cohort/data/01/'
    input_file = f"{mainpath}/1_seg/RV_smooth_flip.vtk"
    output_file = f"{mainpath}/1_seg/RV_smooth_flip.vtk"
    
    # Extract the septum mesh
    septum_mesh = extract_septum_from_rv_mesh(input_file, output_file, septum_tag=3)
    
    if septum_mesh is not None:
        print("\n" + "="*50)
        print("SUCCESS! Septum mesh extracted and saved.")
        print("="*50)
        
        # Optional: Load both meshes for comparison
        rv_mesh = pv.read(input_file)
        
        # Uncomment the line below if you want to visualize
        # visualize_extraction(rv_mesh, septum_mesh)
    
    else:
        print("\n" + "="*50)
        print("FAILED! Could not extract septum mesh.")
        print("="*50)

# Quick one-liner version:
print("\nQUICK ONE-LINER VERSION:")
print("rv_mesh = pv.read('RV_smooth_flip.vtk')")
print("septum_mesh = rv_mesh.threshold(2.5, 3.5, scalars='elemTag')")
print("septum_mesh.save('RV_septum_polar.vtk')")

Loading mesh from: /data_fast/Work/MS_HF_cohort/data/01//1_seg/RV_smooth_flip.vtk
Original mesh:
  Points: 7918
  Cells: 15385
  Point data arrays: ['vtkOriginalPointIds']
  Cell data arrays:  ['elemTag', 'vtkOriginalCellIds']

Using elemTag from: cell_data
Unique elemTag values: [3]
  Tag 3: 15385 cells

Extracting septum mesh...

Final septum mesh:
  Points: 7918
  Cells: 15385
  Extracted cell elemTags: [3]

Saving septum mesh to: /data_fast/Work/MS_HF_cohort/data/01//1_seg/RV_smooth_flip.vtk
✓ Septum mesh saved successfully!

SUCCESS! Septum mesh extracted and saved.

QUICK ONE-LINER VERSION:
rv_mesh = pv.read('RV_smooth_flip.vtk')
septum_mesh = rv_mesh.threshold(2.5, 3.5, scalars='elemTag')
septum_mesh.save('RV_septum_polar.vtk')


In [8]:
network_file=f'{mainpath}/1_seg/final_network_flip.vtk'
CS=pv.read(network_file)
print(CS)
print(CS.cell_data['elemTag'])

# HB=150 
LB = 151
# LV ant=155
# LV sept=153
# LV post = 154
# RB = 152
# RV - 156,157
# LV His PK 101
# RV His PK 102

CS_upper = CS.threshold(value=(152.5, 155.5), scalars='elemTag') 
CS_upper=CS_upper.extract_surface()
CS_upper.save(f'{mainpath}/1_seg/LB_ant_post.vtk')

LB_mesh = CS.threshold(value=(LB-0.5, LB+0.5), scalars='elemTag') 
LB_mesh.save(f'{mainpath}/1_seg/LBB.vtk')


PolyData (0x79809c07c5e0)
  N Cells:    106196
  N Points:   80376
  N Strips:   0
  X Bounds:   -7.083e+01, 2.867e+01
  Y Bounds:   -1.500e+02, -5.102e+01
  Z Bounds:   3.270e+02, 4.363e+02
  N Arrays:   2
[150 150 150 ... 102 102 102]


In [17]:
TV=pv.read(f'{mainpath}/1_seg/TV_annulus-20_flip.vtk')

surf = TV.delaunay_2d()
print(surf)
surf.save(f'{mainpath}/1_seg/TV20_surface.vtk')

PolyData (0x74f330dac5e0)
  N Cells:    91
  N Points:   83
  N Strips:   0
  X Bounds:   -1.348e+01, -7.296e+00
  Y Bounds:   -1.109e+02, -9.792e+01
  Z Bounds:   3.392e+02, 3.870e+02
  N Arrays:   0


In [56]:
import pyvista as pv
file='/data_fast/Work/MS_HF_cohort/data/01//1_seg/sept_AHA.vtk'
file2='/data_fast/Work/MS_HF_cohort/data/01//1_seg/sept_AHA_CSP.stl'

mesh=pv.read(file)
#display(mesh)
CSP=True
septum_mesh=map_AHA_regions(csv_file=None, septum_mesh=mesh, CSP=CSP)
display(mesh)

# CSP regions mapping 
# LBB: 20, LAF: 21, LPF: 22, LSF: 24
# LBB: 20, 41, 42, 44, 63, 65, 66
# LAF: 21, 41, 43, 45, 65, 67
# LPF: 22, 42, 43, 46, 63, 66, 67
# LSF: 24, 44, 45, 46, 65, 66, 67

csp = mesh.point_data["CSP_mapping"]

# Define the mapping (value to assign, list of CSP IDs to match)
fields = {
    "LBB": (20, [20, 41, 42, 44, 63, 65, 66]),
    "LAF": (21, [21, 41, 43, 45, 65, 67]),
    "LPF": (22, [22, 42, 43, 46, 63, 66, 67]),
    "LSF": (24, [24, 44, 45, 46, 65, 66, 67]),
}

# Create each new point_data array
for name, (value, ids) in fields.items():
    arr = np.zeros_like(csp, dtype=int)
    mask = np.isin(csp, ids)   # True where CSP_mapping matches
    arr[mask] = value
    mesh.point_data[name] = arr

mesh = mesh.extract_surface()
#mesh.export(file2)
# plotter=pv.Plotter()
# plotter.add_mesh(mesh)
# plotter.show()
from vtk import vtkSTLWriter

# mesh is a PyVista PolyData object
stl_writer = vtkSTLWriter()
stl_writer.SetFileName(file2)
stl_writer.SetInputData(mesh)  # PyVista PolyData works as vtkPolyData
stl_writer.Write()

[260, 280]
[270, 330]
[225, 270]
[252, 274]


UnstructuredGrid (0x7f9af5b3e140)
  N Cells:    15385
  N Points:   7918
  X Bounds:   -5.222e+01, 3.373e+00
  Y Bounds:   -1.166e+02, -6.719e+01
  Z Bounds:   3.335e+02, 4.166e+02
  N Arrays:   8

1

In [ ]:
np.unique(mesh['CSP_mapping'])
# LBB: 20, LAF: 21, LPF: 22, LSF: 24
# LBB: 20, 41,42,44,65,66
# LAF: 21, 41, 65
# LPF: 22, 42, 46, 66
# LSF: 24, 44, 46, 65, 66

pyvista_ndarray([ 0, 20, 21, 22, 24, 41, 42, 44, 46, 65, 66])

In [58]:
CSP_colors={
        'LBB' : {'color': "#FF86FF", 'txtcol': "#FF4DFF"}, # LB 
        'LAF' : {'color': "#fdb56c", 'txtcol': '#cc6600'}, # LV anterior branch area
        'LPF' : {'color': '#4da6ff', 'txtcol': "#003d7a"}, # LV posterior branch area 
        'LSF' : {'color': '#66b38c', 'txtcol': "#00664f"} # LV posterior branch area 
    }
txtcol=CSP_colors["LAF"]['txtcol']
print(txtcol)

#cc6600


In [35]:
rv_sept_mesh = pv.read(f'{mainpath}/1_seg/RV_septum_polar.vtk') 
display(rv_sept_mesh.bounds)
rv_sept_mesh.translate([10,0,0], inplace=True)  # shifts all points by +1 in x, y, z
display(rv_sept_mesh.bounds)
print(rv_sept_mesh.n_points)  # number of points

BoundsTuple(x_min=-52.22460174560547, x_max=1.3628300428390503, y_min=-116.04499816894531, y_max=-69.34390258789062, z_min=335.5140075683594, z_max=414.2550048828125)

BoundsTuple(x_min=-42.22460174560547, x_max=11.36283016204834, y_min=-116.04499816894531, y_max=-69.34390258789062, z_min=335.5140075683594, z_max=414.2550048828125)

6637
